# Pygame Event Handling Architecture & Design Guide

## 1. Core Architectural Strategy

Pygame gives us two different ways to listen to player input: Event Queue Polling (`pygame.event.get()`) and Direct State Polling (`pygame.key.get_pressed()`).

To build smooth controls without animation glitches or input lag, we use a hybrid approach: Events trigger one-time actions, while Polling handles continuous movement.

```
┌───────────────────────────────────┐
                          │         PLAYER INPUT SYSTEM       │
                          └─────────────────┬─────────────────┘
                                            │
                  ┌─────────────────────────┴─────────────────────────┐
                  │                                                   │
    ┌─────────────▼──────────────┐                      ┌─────────────▼──────────────┐
    │  Event Queue               │                      │  State Polling             │
    │  pygame.event.get()        │                      │  pygame.key.get_pressed()   │
    ├────────────────────────────┤                      ├────────────────────────────┤
    │ • Triggered ONCE per press │                      │ • Checked EVERY FRAME      │
    │ • Ideal for one-shot or    │                      │ • Ideal for continuous     │
    │   toggle actions           │                      │   holding & movement       │
    ├────────────────────────────┤                      ├────────────────────────────┤
    │ Examples:                  │                      │ Examples:                  │
    │ - Meditate (Down Key)      │                      │ - Run Left (Hold Left)     │
    │ - Bruce Lee (Mouse Click)  │                      │ - Run Right (Hold Right)   │
    │ - Quit / Timers            │                      │ - Natural Stopping         │
    └────────────────────────────┘                      └────────────────────────────┘
```

### A. Event Queue Polling (pygame.event.get())

* __Think of it like__: Getting an email or push notification on your phone. It alerts you once when something happens, then sits quietly until the next message arrives.

* __How it works__: Pygame maintains a list (a "queue") of actions that occurred since the last frame. Inside your event loop, you process each item once as it happens.

* Best used for:
    - __One-shot triggers__: Toggling a menu, pausing, clicking on a sprite, or launching a jump.
    - __System & Custom events__: Closing the window (`QUIT`) or handling custom spawn timers (`SPAWN_BOMB_EVENT`).
    - __Targeted Action Postures__: Specific states like entering `MEDITATING` when the Down Arrow is tapped, or `BRUCE_LEE` when the player left-clicks the Ninja.
```
for event in pygame.event.get():
    # Triggered ONCE when the Down Arrow key is first pressed down
    if event.type == pygame.KEYDOWN and event.key == pygame.K_DOWN:
        ninja.current_status = NinjaState.MEDITATING
```

### Direct State Polling (pygame.key.get_pressed())

* __Think of it like__: Looking at a speedometer or holding down a car accelerator pedal. It continuously reports the exact state of every key right now, frame by frame.

* __How it works__: Call `pygame.key.get_pressed()` inside your `update()` method. It returns a snapshot list of every key on the keyboard, marked as either `True` (currently held down) or `False` (not held down).

* __Best used for:__
    - __Continuous spatial movement__: Holding down K_LEFT or K_RIGHT to make the Ninja run across the screen at 60 FPS.
    - __Animation state synchronization__: Automatically switching to RUNNING_LEFT while holding Left, RUNNING_RIGHT while holding Right, and cleanly returning to STANDING as soon as keys are released.

```
keys = pygame.key.get_pressed()

# Checked EVERY FRAME (60 times per second)
if keys[pygame.K_LEFT] and not keys[pygame.K_RIGHT]:
    ninja.rect.left = max(0, ninja.rect.left - ninja.vel)
    ninja.current_status = NinjaState.RUNNING_LEFT
elif keys[pygame.K_RIGHT] and not keys[pygame.K_LEFT]:
    ninja.rect.right = min(screen_width, ninja.rect.right + ninja.vel)
    ninja.current_status = NinjaState.RUNNING_RIGHT
else:
    ninja.current_status = NinjaState.STANDING
```

#### Step-by-Step State Transition Breakdown:

Unlike discrete event handlers that rely on a `KEYUP` event to tell the game when a key was let go, state polling evaluates key states every single frame:

1. While holding `K_LEFT`: `keys[pygame.K_LEFT]` evaluates to `True`, continuously moving the sprite left and setting `self.current_status = NinjaState.RUNNING_LEFT`.
2. The instant `K_LEFT` is released:
    - On the very next frame tick (within 16ms at 60 FPS), `update()` executes.
    - `keys[pygame.K_LEFT]` now evaluates to False.
    - Both directional checks fail:
        - `if (False and True) $\rightarrow$ False`
        - `elif (False and True) $\rightarrow$ False`
    - Execution drops directly into the `else` block:
    ```
    else:
    self.current_status = NinjaState.STANDING
    ```
3. When both `K_LEFT` and `K_RIGHT` are held down simultaneously:
    - `keys[pygame.K_LEFT]` is True, but not `keys[pygame.K_RIGHT]` evaluates to False.
    - Both `if and elif` checks evaluate to `False`.
    - Execution falls safely into the `else` block, stopping movement and restoring `STANDING`stance instantly.

    __Which Input Method Should I Use?__

| Question to Ask Yourself | Mechanism | Method | Example in Game |
|---|---|---|---|
| **Did the player just perform an action once?** | Event Queue | `pygame.event.get()` | Clicking the Ninja for Bruce Lee pose (`MOUSEBUTTONDOWN`), or spawning a bomb when a timer fires (`USEREVENT`) |
| **Is the player holding down a button to keep moving?** | State Polling | `pygame.key.get_pressed()` | Holding `K_LEFT` or `K_RIGHT` to run smoothly across the screen every frame |

## 2. Event Categorization & Flow Map

The game engine separates input processing into four distinct functional streams: three routed through the Pygame Event Queue (`pygame.event.get()`), and one handled via Direct State Polling (`pygame.key.get_pressed()`).

```
MAIN GAME LOOP
                                        │
           ┌────────────────────────────┴────────────────────────────┐
           │                                                         │
┌──────────▼─────────────────────────┐               ┌───────────────▼───────────────┐
│        Pygame Event Queue          │               │     Direct Keyboard Polling   │
│        pygame.event.get()          │               │    pygame.key.get_pressed()   │
└──────────────────┬─────────────────┘               └───────────────┬───────────────┘
                   │                                                 │
   ┌───────────────┼───────────────┐                                 │
   │               │               │                                 │
┌──▼─────────┐  ┌──▼─────────┐  ┌──▼─────────┐                    ┌──▼─────────┐
│  Stream 1  │  │  Stream 2  │  │  Stream 3  │                    │  Stream 4  │
│   System   │  │Custom Timer│  │  Discrete  │                    │ Continuous │
│  (QUIT)    │  │ (Spawners) │  │  (Action)  │                    │ (Movement) │
└─────┬──────┘  └─────┬──────┘  └─────┬──────┘                    └─────┬──────┘
      │               │               │                                 │
┌─────▼──────┐  ┌─────▼──────┐  ┌─────▼──────┐                    ┌─────▼──────┐
│  running   │  │ Add Sprites│  │ninja.handle│                    │ninja.update│
│  = False   │  │  to Group  │  │_events()   │                    │ (Movement) │
└────────────┘  └────────────┘  └────────────┘                    └────────────┘
```

__Stream Definitions__

1. __Stream 1: System Control Events__ (`pygame.QUIT`) — Intercepts window/OS-level close commands in `main.py` to stop the loop.
2. __Stream 2: Custom Timed Events__ (`pygame.USEREVENT + N`) — Receives timer triggers (`pygame.time.set_timer`) to spawn falling items at fixed intervals.
3. __Stream 3__: Discrete Player Inputs (`ninja.handle_events()`) — Captures single-press and single-click triggers (`MEDITATING` via Down arrow, `BRUCE_LEE` via mouse click).
4. __Stream 4__: Continuous State Inputs (ninja.update()) — Polls keyboard state every frame for horizontal movement ($\leftarrow$ / $\rightarrow$) and directional animation synchronization.



## 3. Deep Dive: Ninja Input & Animation State Machine

The `Ninja` class uses a state machine represented by the `NinjaState Enum`. Input processing is split cleanly between `handle_events()` and `update()`.

```
from pygame.event import Event
import pygame
from ninja_config import NinjaState


class Ninja(pygame.sprite.Sprite):
    MAX_LIVES = 5

    def __init__(self, ninja_rect: pygame.Rect, status_file_dict: dict, vel: int = 5):
        super().__init__()
        self.rect = ninja_rect
        self.status_surface_dict = self._create_status_surface_dic(status_file_dict)
        self.default_status = NinjaState.STANDING
        self.current_status = NinjaState.STANDING
        self.vel = vel
        self.num_of_lives = Ninja.MAX_LIVES
        self.score = 0

    @property
    def image(self) -> pygame.Surface:
        return self.status_surface_dict[self.current_status]

    @property
    def is_alive(self) -> bool:
        """Read-only property serving as the single source of truth for player life status."""
        return self.num_of_lives > 0 and self.current_status != NinjaState.DEAD

    # -------------------------------------------------------------------------
    # 1. DISCRETE EVENT HANDLER (Triggers, Postures, Specific Key Releases)
    # -------------------------------------------------------------------------
    def handle_events(self, event: Event):
        if not self.is_alive:
            return

        # Meditating Pose: Strictly tied to DOWN arrow press and release
        if event.type == pygame.KEYDOWN and event.key == pygame.K_DOWN:
            self.current_status = NinjaState.MEDITATING

        elif event.type == pygame.KEYUP and event.key == pygame.K_DOWN:
            if self.current_status == NinjaState.MEDITATING:
                self.current_status = NinjaState.STANDING

        # Bruce Lee Pose: Strictly tied to Left Mouse Click (button 1) on Sprite
        elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.rect.collidepoint(event.pos):
                self.current_status = NinjaState.BRUCE_LEE

        elif event.type == pygame.MOUSEBUTTONUP and event.button == 1:
            if self.current_status == NinjaState.BRUCE_LEE:
                self.current_status = NinjaState.STANDING

    # -------------------------------------------------------------------------
    # 2. CONTINUOUS STATE & MOVEMENT UPDATER (Runs every frame inside loop)
    # -------------------------------------------------------------------------
    def update(self, screen_width: int):
        if not self.is_alive:
            return

        # Action Guard: Stationary action postures block directional movement
        if self.current_status in (NinjaState.MEDITATING, NinjaState.BRUCE_LEE):
            return

        keys = pygame.key.get_pressed()
        moving_left = keys[pygame.K_LEFT]
        moving_right = keys[pygame.K_RIGHT]

        # Opposite Direction Chord Resolution
        if moving_left and not moving_right:
            self.rect.left = max(0, self.rect.left - self.vel)
            self.current_status = NinjaState.RUNNING_LEFT

        elif moving_right and not moving_left:
            self.rect.right = min(screen_width, self.rect.right + self.vel)
            self.current_status = NinjaState.RUNNING_RIGHT

        else:
            # Neutral stance when no movement keys (or both) are held
            # Automatically catches key releases on the very next frame tick!
            self.current_status = NinjaState.STANDING
```

## 4. Key Implementation Rules & Edge-Case Guards

### Rule 1: Strict Key and Button Matching

__Rule 1: Strict Key and Button Matching__

* __Problem__: Listening for generic `KEYUP` or `MOUSEBUTTONUP` events causes unrelated inputs (like releasing `Space` or right-clicking) to cancel active postures.
* __Solution__: Match both `event.type` AND `event.key` / `event.button` strictly:

```
elif event.type == pygame.KEYUP and event.key == pygame.K_DOWN:
```

__Rule 2: Synchronize Running Animation inside `update()`__

* __Problem__: Managing running animations inside `handle_events()` causes desynchronization when `KEYUP` events are missed or overlapping keys are pressed.
*__Solution__: Derive running states (`RUNNING_LEFT`, `RUNNING_RIGHT`, `STANDING`) directly from `pygame.key.get_pressed()` every frame inside `update()`.

__Rule 3: Opposite-Direction Chord Handling__

* __Problem__: Pressing $\leftarrow$ and $\rightarrow$ simultaneously causes positional jitter or conflicting animation updates.
* __Solution__: Use exclusive boolean evaluation (`moving_left` and `not moving_right`) to fall back safely to `STANDING` when opposing inputs cancel out.

__Rule 4: Action Posture Guard__: 

* __Problem__: Pressing movement keys while in a stationary stance (`MEDITATING`, `BRUCE_LEE`) breaks character posture.
* __Solution__: Early return in `update()` when active in a stationary posture:

```
if self.current_status in (NinjaState.MEDITATING, NinjaState.BRUCE_LEE):
    return
```

## 5. Summary Reference Checklist


| Feature / Action | Stream | Mechanism | Code Location | Key Design Note |
|---|---|---|---|---|
| **Quit Game** | Stream 1 | `pygame.QUIT` | `main.py` | Sets loop condition `running = False`. |
| **Spawn Items** | Stream 2 | `USEREVENT + N` | `main.py` | Triggered periodically by `pygame.time.set_timer`. |
| **Meditate** | Stream 3 | `KEYDOWN` / `KEYUP` | `Ninja.handle_events()` | Filtered strictly for `pygame.K_DOWN`. |
| **Bruce Lee Pose** | Stream 3 | `MOUSEBUTTONDOWN` | `Ninja.handle_events()` | Filtered strictly for `event.button == 1` and `collidepoint`. |
| **Horizontal Movement** | Stream 4 | `get_pressed()` | `Ninja.update()` | Bounded by screen width limits. |
| **Running Animation** | Stream 4 | `get_pressed()` | `Ninja.update()` | Evaluates direction state continuously each frame; else catches key releases. |